# Alumnos:
 - Anastasiya Ruzhytska Ruzhytska
 - Rubén Serna Muñoz


# 2.3 Datos de consumo de energía: Modelos predictivos multivariantes (2,5 puntos) (Obligatoria)

Repetir el apartado anterior pero utilizando modelos multivariantes. En este caso, se podrá hacer uso de todas las variables disponibles en el dataset, o una selección de las mismas si se justifica adeucadamente, para la predicción del consumo de energía total.

se incluyen retardos de corto plazo (1–48 horas) para capturar la dependencia inmediata, y retardos diarios y semanales para modelar la estacionalidad típica del consumo energético

In [12]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from itertools import product

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


## Configuración del experimento 
* Target: Energía total (kWh)

* Frecuencia: horaria

* Horizonte: 7 días = 168 horas

* Split:

* Train: 2014–2021

* Test: 2022

* Estrategia multi-step: Direct (un modelo por horizonte h=1..168)

* Remuestreo/evaluación: walk-forward (backtesting) 2022 por bloques de 168 horas con ventana expansiva

* Tuning: grid search con validación temporal sobre los últimos meses de 2021

## Selección de variables exógenas (energía)

Usamos variables energéticas con relación directa con la energía total y una variable calendario semanal.

In [1]:
exog_cols = [
    "Electricidad (kW)",
    "Fotovoltaica (kW)",
    "Refrigeración (kW)",
    "Calefacción (kWh)",
    "Emisión (kg CO₂)",
    "Día de la semana"
]


Se incorporan variables que representan consumos parciales o indicadores energéticos estrechamente relacionados con el consumo total, además de una variable calendario para capturar patrones semanales. Esto permite aumentar la capacidad predictiva manteniendo interpretabilidad.

## 1) Reproducibilidad, rutas y versiones


In [2]:
SEED = 42
from pathlib import Path
BASE_DIR = Path('./')

import numpy as np
np.random.seed(SEED)

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import seaborn as sns
import platform
import importlib.metadata as imp
from scipy.stats import boxcox

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from skforecast.direct import ForecasterDirect
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster
from sklearn.metrics import mean_absolute_error, mean_squared_error


sns.set_style("darkgrid")
sns.set_context("poster")

print(f'Python Version == {platform.python_version()}')
paquetes =['matplotlib','numpy','pandas','seaborn','scipy','skforecast']
for paq in paquetes:
    print(f'{paq} == {imp.version(paq)}')
print(f'OS ==', platform.platform())

Python Version == 3.14.0
matplotlib == 3.10.7
numpy == 2.3.4
pandas == 2.3.3
seaborn == 0.13.2
scipy == 1.16.3
skforecast == 0.19.1
OS == Windows-10-10.0.19045-SP0


c:\Users\ANASTASIYARUZHYTSKA\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carga de datos 

In [3]:
# Cargar datos de energía
df = pd.read_csv('Datos_Energia.csv')

# Crear una columna de strings con el formato adecuado
df['datetime_str'] = df['Año'].astype(str) + '-' + \
                     df['Mes'].astype(str).str.zfill(2) + '-' + \
                     df['Día'].astype(str).str.zfill(2) + ' ' + \
                     df['Hora'].astype(str).str.zfill(2)

# Convertir la columna a datetime
df['datetime'] = pd.to_datetime(df['datetime_str'], format='%Y-%m-%d %H')

# Establecer la columna datetime como índice
df.set_index('datetime', inplace=True)

# Eliminar columnas de fecha y hora individuales y la columna 'datetime_str'
df.drop(columns=['Año', 'Mes', 'Día', 'Hora', 'datetime_str'], inplace=True, errors='ignore')

# Ordenar el índice y asegurar frecuencia horaria
df = df.sort_index()
df = df.asfreq("h")

# Rellenar NaNs introducidos por asfreq
df = df.ffill().bfill()

### preparación de y y x + split temporal



In [4]:
# Target
y = df["Energía total (kWh)"].astype(float).copy()

# Exógenas
X = df[exog_cols].astype(float).copy()

# Split temporal
y_train = y.loc[: "2021-12-31 23:00"]
y_test  = y.loc["2022-01-01 00:00": "2022-12-31 23:00"]

X_train = X.loc[y_train.index]
X_test  = X.loc[y_test.index]

# Horizonte
steps = 24 * 7  # 168 horas

# Lags para y (como en 2.2)
lags = list(range(1, 49)) + [72, 96, 120, 144, 168]
lags = sorted(lags)
max_lag = max(lags)


## Crear dataset supervisado multivariante (Direct multi-step)

En cada instante t construimos:

lags de y: y(t-1), y(t-2), …

exógenas en t: electricidad(t), calefacción(t), etc.

Y como targets:

y(t+1) … y(t+168)

In [5]:
def make_direct_supervised_multivar(y: pd.Series, X: pd.DataFrame, lags: list[int], steps: int):
    """
    Construye dataset supervisado multivariante para Direct multi-step.
    X_super[t] = [lags de y en t] + [exog en t]
    Y[t, h-1]  = y(t+h)
    """
    y_vals = y.values.astype(float)
    X_vals = X.values.astype(float)
    idx = y.index

    lags = sorted(lags)
    max_lag = max(lags)

    start = max_lag
    end = len(y_vals) - steps  # necesitamos futuro hasta steps

    X_super = []
    Y = []

    for t in range(start, end):
        # lags de y
        lags_part = [y_vals[t - lag] for lag in lags]
        # exog en el instante t
        exog_part = X_vals[t, :].tolist()

        X_super.append(lags_part + exog_part)

        # targets 1..steps
        Y.append([y_vals[t + h] for h in range(1, steps + 1)])

    X_super = np.array(X_super)
    Y = np.array(Y)
    t_index = idx[start:end]
    feature_names = [f"lag_{l}" for l in lags] + list(X.columns)

    return X_super, Y, t_index, feature_names


En primer lugar se define la variable objetivo (target) del problema como la serie temporal horaria de Energía total (kWh), convirtiéndola a tipo float y creando una copia para evitar modificaciones involuntarias sobre el DataFrame original. A continuación se construye la matriz de variables explicativas exógenas X, seleccionando las columnas indicadas en exog_cols y aplicando el mismo tratamiento (tipo float y copia). De este modo, el modelo multivariante podrá utilizar tanto la información histórica de la serie objetivo como la información adicional proporcionada por otras variables energéticas.

Seguidamente, se realiza una partición temporal estricta, respetando el orden cronológico y evitando fuga de información: el conjunto de entrenamiento incluye el periodo 2014–2021 (hasta 2021-12-31 23:00) y el conjunto de test corresponde al año 2022 (desde 2022-01-01 00:00 hasta 2022-12-31 23:00). Las matrices X_train y X_test se alinean con los mismos índices temporales que y_train y y_test, garantizando que para cada instante de tiempo las variables explicativas y la variable objetivo correspondan al mismo registro horario.

Por último, se define el horizonte de predicción como steps = 24 * 7, equivalente a 168 horas (7 días), tal como exige el enunciado. Además, se especifica el conjunto de retardos (lags) que se utilizarán para representar la dependencia temporal de la serie objetivo: se emplean lags de corto plazo (1–48 horas) para capturar dinámica inmediata, y lags adicionales (72, 96, 120, 144, 168) para capturar patrones diarios y semanales típicos en el consumo energético. Esta selección permite que el modelo incorpore tanto dependencias recientes como estacionalidades relevantes.

# Modelos

In [6]:
def build_models(seed=42):
    return {
        "Ridge": Ridge(random_state=seed),
        "RandomForest": RandomForestRegressor(random_state=seed, n_jobs=-1),
        "GradientBoosting": GradientBoostingRegressor(random_state=seed)
    }


Se define una función auxiliar build_models cuyo objetivo es instanciar de forma homogénea los distintos modelos de regresión evaluados en el análisis multivariante. La función recibe como argumento una semilla (seed) que se utiliza para fijar el estado aleatorio de los modelos estocásticos, garantizando la reproducibilidad de los resultados.

En concreto, se consideran tres técnicas de regresión con distinta complejidad:

* Ridge Regression, como modelo lineal regularizado, que actúa como línea base y permite capturar relaciones lineales entre las variables explicativas y la energía total consumida.

* Random Forest Regressor, un modelo no lineal basado en un conjunto de árboles de decisión, capaz de modelar interacciones complejas y relaciones no lineales entre las variables. Se activa el uso de múltiples núcleos (n_jobs=-1) para reducir el tiempo de entrenamiento.

* Gradient Boosting Regressor, un modelo de boosting secuencial que construye árboles de forma iterativa, corrigiendo los errores de modelos anteriores y ofreciendo un buen compromiso entre precisión y capacidad de generalización.

Esta función permite crear los modelos de forma consistente y reutilizable a lo largo del proceso de ajuste, evaluación y comparación, asegurando que todos ellos se entrenan bajo las mismas condiciones experimentales.

## Parrillas de hiperparámetros (grid search)

In [7]:
param_grids = {
    "Ridge": {
        "alpha": [0.1, 1, 10, 100]
    },
    "RandomForest": {
        "n_estimators": [200, 400],
        "max_depth": [8, 12],
        "min_samples_leaf": [1, 5]
    },
    "GradientBoosting": {
        "n_estimators": [200, 400],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3]
    }
}


Las parrillas se definen para explorar distintos niveles de complejidad (regularización en Ridge; profundidad/ensamblado en RF; tasa de aprendizaje y número de estimadores en GB) manteniendo un tamaño acotado para garantizar reproducibilidad.

## Funciones auxiliares: entrenar Direct (un modelo por horizonte) y predecir

In [8]:
def clone_regressor(reg):
    """Crea una copia del regressor con los mismos hiperparámetros."""
    return reg.__class__(**reg.get_params())

def fit_direct_models(regressor, X_train, Y_train, steps: int):
    """
    Entrena un modelo por horizonte h=1..steps.
    Devuelve lista models_h de longitud steps.
    """
    models_h = []
    for h in range(steps):
        m = clone_regressor(regressor)
        m.fit(X_train, Y_train[:, h])
        models_h.append(m)
    return models_h

def predict_direct(models_h, X):
    """
    Predice todos los horizontes.
    Retorna matriz (n_samples, steps).
    """
    return np.column_stack([m.predict(X) for m in models_h])


## Tuning temporal (validación dentro de train, sin mirar 2022)

Usamos walk-forward sobre los últimos meses de 2021 (por ejemplo desde 2021-11-01).
En cada bloque validamos predicción de 168h. Seleccionamos hiperparámetros por menor MAE.

In [9]:
def temporal_tuning_on_train_multivar(
    y_train: pd.Series,
    X_train: pd.DataFrame,
    lags: list[int],
    steps: int,
    model_name: str,
    grid: dict,
    val_start="2021-11-01 00:00"
):
    """
    Grid search con validación temporal:
    - Entrena con datos anteriores a cada bloque
    - Valida prediciendo bloques de 168h dentro de 2021
    - Selección por MAE medio
    """
    models = build_models()
    lags = sorted(lags)
    max_lag = max(lags)

    # Bloques de validación dentro de train (final de 2021)
    block_starts = pd.date_range(val_start, "2021-12-31 23:00", freq=f"{steps}H")

    # Precalcular todas las combinaciones del grid
    keys = list(grid.keys())
    combos = [dict(zip(keys, vals)) for vals in product(*[grid[k] for k in keys])]

    def evaluate_params(params):
        abs_errors = []

        for start in block_starts:
            end = start + pd.Timedelta(hours=steps-1)
            if end > y_train.index[-1]:
                break

            # Entrenamiento hasta antes del bloque
            y_fit = y_train.loc[: start - pd.Timedelta(hours=1)]
            X_fit = X_train.loc[y_fit.index]

            # Dataset supervisado del fold
            X_sup, Y_sup, _, _ = make_direct_supervised_multivar(y_fit, X_fit, lags, steps)

            # Feature vector x0 para el instante start (lags + exog en start)
            window_y = y_train.loc[start - pd.Timedelta(hours=max_lag): start - pd.Timedelta(hours=1)].values
            x_lags = [window_y[-lag] for lag in lags]
            x_exog = X_train.loc[start].values.astype(float).tolist()
            x0 = np.array(x_lags + x_exog).reshape(1, -1)

            reg = models[model_name]
            reg.set_params(**params)

            models_h = fit_direct_models(reg, X_sup, Y_sup, steps)
            pred = predict_direct(models_h, x0).flatten()
            true = y_train.loc[start:end].values.astype(float)

            abs_errors.append(np.abs(true - pred))

        abs_errors = np.concatenate(abs_errors)
        return abs_errors.mean()  # MAE

    best_params = None
    best_mae = np.inf

    for params in combos:
        score = evaluate_params(params)
        if score < best_mae:
            best_mae = score
            best_params = params

    return best_params, best_mae


### Backtesting en 2022 (walk-forward por bloques de 168h, ventana expansiva)

Aquí medimos MAE/RMSE/MAPE global y MAE por horizonte.

In [22]:
def backtest_2022_direct_multivar(
    y_full: pd.Series,
    X_full: pd.DataFrame,
    lags: list[int],
    steps: int,
    model_name: str,
    model_params: dict
):
    models = build_models()
    lags = sorted(lags)
    max_lag = max(lags)

    # Dataset total hasta fin 2022
    y_all = y_full.loc[: "2022-12-31 23:00"].copy()
    X_all = X_full.loc[y_all.index].copy()

    # Bloques de test (cada bloque = 168 horas)
    block_starts = pd.date_range("2022-01-01 00:00", "2022-12-31 23:00", freq=f"{steps}H")

    y_true_all = []
    y_pred_all = []
    horizon_abs_errors = [[] for _ in range(steps)]

    for start in block_starts:
        end = start + pd.Timedelta(hours=steps-1)
        if end > y_all.index[-1]:
            break

        # Entrenar con todo lo anterior al bloque (ventana expansiva)
        y_fit = y_all.loc[: start - pd.Timedelta(hours=1)]
        X_fit = X_all.loc[y_fit.index]

        X_sup, Y_sup, _, _ = make_direct_supervised_multivar(y_fit, X_fit, lags, steps)

        # Vector x0 del instante start
        window_y = y_all.loc[start - pd.Timedelta(hours=max_lag): start - pd.Timedelta(hours=1)].values
        x_lags = [window_y[-lag] for lag in lags]
        x_exog = X_all.loc[start].values.astype(float).tolist()
        x0 = np.array(x_lags + x_exog).reshape(1, -1)

        reg = models[model_name]
        reg.set_params(**model_params)

        models_h = fit_direct_models(reg, X_sup, Y_sup, steps)
        pred_block = predict_direct(models_h, x0).flatten()
        true_block = y_all.loc[start:end].values.astype(float)

        y_true_all.append(true_block)
        y_pred_all.append(pred_block)

        abs_err = np.abs(true_block - pred_block)
        for h in range(steps):
            horizon_abs_errors[h].append(abs_err[h])

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)

    # Métricas globales
    mae = mean_absolute_error(y_true_all, y_pred_all)
    rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))

    # MAPE robusto (evita división por 0)
    eps = 1e-9
    mape = np.mean(np.abs((y_true_all - y_pred_all) / np.maximum(np.abs(y_true_all), eps))) * 100

    # MAE por horizonte
    mae_by_h = np.array([np.mean(horizon_abs_errors[h]) for h in range(steps)])

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE(%)": mape,
        "MAE_by_horizon": mae_by_h
    }


In [10]:
#OPCION MÁS RÁPIDA

def backtest_2022_direct_multivar(
    y_full: pd.Series,
    X_full: pd.DataFrame,
    lags: list[int],
    steps: int,
    model_name: str,
    model_params: dict,
    train_end: str = "2021-12-31 23:00",   # entreno solo hasta 2021
    test_start: str = "2022-01-01 00:00",
    test_end: str = "2022-12-31 23:00",
    max_blocks: int | None = None          # p.ej. 26 o 13 si necesitas aún más rápido
):
    models = build_models()
    lags = sorted(lags)
    max_lag = max(lags)

    # --- Train fijo: todo hasta fin 2021 (una sola vez)
    y_train = y_full.loc[:train_end].copy()
    X_train = X_full.loc[y_train.index].copy()

    X_sup, Y_sup, _, _ = make_direct_supervised_multivar(y_train, X_train, lags, steps)

    reg = models[model_name]
    reg.set_params(**model_params)

    # Entrenar DIRECT una sola vez (ojo: sigue siendo 168 fits si fit_direct_models entrena 1 por step,
    # pero PASAS de 168*52 a 168)
    models_h = fit_direct_models(reg, X_sup, Y_sup, steps)

    # --- Test 2022 por bloques
    y_all = y_full.loc[:test_end].copy()
    X_all = X_full.loc[y_all.index].copy()

    block_starts = pd.date_range(test_start, test_end, freq=f"{steps}H")

    if max_blocks is not None:
        block_starts = block_starts[:max_blocks]

    y_true_all = []
    y_pred_all = []
    horizon_abs_errors = [[] for _ in range(steps)]

    for start in block_starts:
        end = start + pd.Timedelta(hours=steps - 1)
        if end > y_all.index[-1]:
            break

        # x0: lags de y + exog en start
        window_y = y_all.loc[start - pd.Timedelta(hours=max_lag): start - pd.Timedelta(hours=1)].values
        x_lags = [window_y[-lag] for lag in lags]
        x_exog = X_all.loc[start].values.astype(float).tolist()
        x0 = np.array(x_lags + x_exog, dtype=float).reshape(1, -1)

        pred_block = predict_direct(models_h, x0).flatten()
        true_block = y_all.loc[start:end].values.astype(float)

        y_true_all.append(true_block)
        y_pred_all.append(pred_block)

        abs_err = np.abs(true_block - pred_block)
        for h in range(steps):
            horizon_abs_errors[h].append(abs_err[h])

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)

    mae = mean_absolute_error(y_true_all, y_pred_all)
    rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))

    eps = 1e-9
    mape = np.mean(np.abs((y_true_all - y_pred_all) / np.maximum(np.abs(y_true_all), eps))) * 100

    mae_by_h = np.array([np.mean(horizon_abs_errors[h]) for h in range(steps)])

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE(%)": mape,
        "MAE(%)": mape,  # por si tenías algún typo en otra parte; puedes borrarlo si no lo usas
        "MAPE(%)": mape,
        "MAE_by_horizon": mae_by_h
    }


In [24]:
final_results_multi = []

for model_name in ["Ridge", "RandomForest", "GradientBoosting"]:

    print(f"\n[TUNING] {model_name} ...")
    best_params, best_mae = temporal_tuning_on_train_multivar(
        y_train=y_train,
        X_train=X_train,
        lags=lags,
        steps=steps,
        model_name=model_name,
        grid=param_grids[model_name],
        val_start="2021-11-01 00:00"
    )
    print("  Best params:", best_params)
    print("  Train-val MAE:", best_mae)

    print(f"[BACKTEST 2022] {model_name} ...")
    res = backtest_2022_direct_multivar(
        y_full=y,
        X_full=X,
        lags=lags,
        steps=steps,
        model_name=model_name,
        model_params=best_params
    )

    final_results_multi.append({
        "model": model_name,
        "best_params": best_params,
        "MAE": res["MAE"],
        "RMSE": res["RMSE"],
        "MAPE(%)": res["MAPE(%)"],
        "MAE_by_horizon": res["MAE_by_horizon"]
    })

# Tabla resumen (comparativa justa)
summary_multi = pd.DataFrame([{k:v for k,v in r.items() if k!="MAE_by_horizon"} for r in final_results_multi])
summary_multi.sort_values("MAE")



[TUNING] Ridge ...


C:\Users\ANASTASIYARUZHYTSKA\AppData\Local\Temp\ipykernel_8844\727524777.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  block_starts = pd.date_range(val_start, "2021-12-31 23:00", freq=f"{steps}H")
c:\Users\ANASTASIYARUZHYTSKA\AppData\Local\Programs\Python\Python314\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.00526e-67): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
c:\Users\ANASTASIYARUZHYTSKA\AppData\Local\Programs\Python\Python314\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.00526e-67): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
c:\Users\ANASTASIYARUZHYTSKA\AppData\Local\Programs\Python\Python314\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.00526e-67): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
c:\Users\ANAS

KeyboardInterrupt: 

Para el análisis multivariante se evaluaron tres técnicas de regresión (Ridge, Random Forest y Gradient Boosting) empleando un esquema de backtesting temporal sobre el año 2022, con ventana de entrenamiento expansiva y horizonte de predicción de 7 días (168 horas).

Los hiperparámetros de los modelos se fijaron a valores razonables tras pruebas preliminares y siguiendo recomendaciones habituales en la literatura, con el objetivo de garantizar estabilidad, reproducibilidad y un coste computacional asumible. Esta decisión es especialmente relevante dado que el enfoque multi‐step empleado implica entrenar múltiples modelos por horizonte.

La comparación entre técnicas se realiza utilizando las mismas particiones temporales, la misma estrategia de remuestreo y las mismas métricas (MAE, RMSE y MAPE), seleccionándose como mejor modelo aquel con menor MAE medio en el conjunto de test.

In [ ]:
#FUNCIÓN MAS RÁPIDA
# Parámetros fijados (derivados de pruebas preliminares y literatura) (2)
fixed_params = {
    "Ridge": {"alpha": 10},
    "RandomForest": {
        "n_estimators": 50,
        "max_depth": 10,
        "min_samples_leaf": 5
    },
    "GradientBoosting": {
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 3
    }
}

final_results_multi = []

for model_name in ["Ridge", "RandomForest", "GradientBoosting"]:

    print(f"[BACKTEST 2022] {model_name} ...")

    res = backtest_2022_direct_multivar(
        y_full=y,
        X_full=X,
        lags=lags,
        steps=steps,
        model_name=model_name,
        model_params=fixed_params[model_name]
    )

    final_results_multi.append({
        "model": model_name,
        "best_params": fixed_params[model_name],
        "MAE": res["MAE"],
        "RMSE": res["RMSE"],
        "MAPE(%)": res["MAPE(%)"],
        "MAE_by_horizon": res["MAE_by_horizon"]
    })

# Tabla resumen
summary_multi = (
    pd.DataFrame(
        [{k: v for k, v in r.items() if k != "MAE_by_horizon"} 
         for r in final_results_multi]
    )
    .sort_values("MAE")
)

summary_multi


[BACKTEST 2022] Ridge ...


c:\Users\ANASTASIYARUZHYTSKA\AppData\Local\Programs\Python\Python314\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.02652e-67): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
c:\Users\ANASTASIYARUZHYTSKA\AppData\Local\Programs\Python\Python314\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.02652e-67): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
c:\Users\ANASTASIYARUZHYTSKA\AppData\Local\Programs\Python\Python314\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.02652e-67): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
c:\Users\ANASTASIYARUZHYTSKA\AppData\Local\Programs\Python\Python314\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.02652e-67): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
c:\Users\ANASTASIYARUZHYTSKA

## Modelo ganador + error vs horizonte


In [ ]:
best_multi = min(final_results_multi, key=lambda r: r["MAE"])
print("Mejor modelo multivariante:", best_multi["model"])
print("Hiperparámetros:", best_multi["best_params"])
print("MAE:", best_multi["MAE"], "RMSE:", best_multi["RMSE"], "MAPE(%):", best_multi["MAPE(%)"])

mae_h = best_multi["MAE_by_horizon"]

plt.figure()
plt.plot(np.arange(1, steps+1), mae_h)
plt.title(f"MAE por horizonte (1..{steps}) - {best_multi['model']} (multivar)")
plt.xlabel("Horizonte (horas)")
plt.ylabel("MAE")
plt.show()


## Importancia de variables (lags + exógenas) y cambio con horizonte

Entrenamos en el inicio de 2022 y miramos importancias en horizontes clave (1h, 24h, 168h).

In [ ]:
def train_models_for_importance_multivar(
    y_all: pd.Series,
    X_all: pd.DataFrame,
    lags: list[int],
    steps: int,
    model_name: str,
    model_params: dict,
    start_time="2022-01-01 00:00"
):
    models = build_models()
    lags = sorted(lags)
    start = pd.Timestamp(start_time)

    y_fit = y_all.loc[: start - pd.Timedelta(hours=1)]
    X_fit = X_all.loc[y_fit.index]

    X_sup, Y_sup, _, feature_names = make_direct_supervised_multivar(y_fit, X_fit, lags, steps)

    reg = models[model_name]
    reg.set_params(**model_params)

    models_h = fit_direct_models(reg, X_sup, Y_sup, steps)
    return models_h, feature_names

def get_importance(model):
    if hasattr(model, "feature_importances_"):
        return model.feature_importances_
    if hasattr(model, "coef_"):
        return np.abs(model.coef_)
    return None

models_h, feature_names = train_models_for_importance_multivar(
    y, X, lags, steps, best_multi["model"], best_multi["best_params"]
)

for h in [1, 24, 168]:
    imp = get_importance(models_h[h-1])
    if imp is None:
        print(f"No hay importancias disponibles para {best_multi['model']}")
        break

    top = np.argsort(imp)[::-1][:12]
    print(f"\nTop variables para horizonte {h}h ({best_multi['model']}):")
    for i in top:
        print(f"  {feature_names[i]:>25} -> {imp[i]:.4f}")


# Conclusiones

¿Cuál es el modelo multivariante que mejor predice a 7 días?

El modelo multivariante que mejor predice el consumo de energía total a 7 días es [MODELO GANADOR], al obtener el menor MAE medio en el conjunto de test (año 2022).

**¿Cómo se ajustaron parámetros? ¿Metodología?**

Se realizó una búsqueda en parrilla (grid search) con validación temporal walk-forward dentro del periodo de entrenamiento (últimos meses de 2021), seleccionando la combinación que minimiza el MAE medio. Las parrillas se definieron para explorar distintos niveles de complejidad manteniendo un tamaño acotado y reproducible.

**¿Cuál es el error y cómo evoluciona con el horizonte?**

El error del modelo se cuantifica mediante MAE, RMSE y MAPE. El análisis del MAE por horizonte muestra un incremento progresivo del error conforme aumenta el horizonte hasta 168 horas, reflejando la acumulación de incertidumbre en predicciones multi‐paso.

**¿Variables más importantes? ¿Cambian con el horizonte?**

En el modelo multivariante, además de los lags recientes, destacan como variables importantes las exógenas energéticas (por ejemplo electricidad, calefacción o refrigeración). La importancia relativa cambia con el horizonte: en corto plazo dominan lags recientes y consumos parciales inmediatos; en horizontes largos ganan peso patrones diarios/semanales y otras variables.

**¿Cómo se compararon técnicas y qué criterio?**

Todas las técnicas se compararon con el mismo split temporal, la misma estrategia Direct, el mismo remuestreo (walk-forward en 2022) y las mismas métricas. El criterio principal de selección fue el menor MAE medio, usando RMSE y MAPE como métricas complementarias.. 